In [ ]:
import pandas as pd
from pathlib import Path
from datetime import date, datetime

import sys
sys.path.insert(0, str(Path("../src").resolve()))
import ado_reporting as cl
import daily_charge as dc
import data_validation as dv
import excel_import as xi
import utilisation as ut

pd.options.display.max_columns = None  # type: ignore
pd.options.display.float_format = "{:,.2f}".format


In [ ]:
# ==== USER CONFIG ====
project = "MarketRMS"

start_date = date(2026, 3, 1)  # <- report generated since that date

# today = date.today()
today = "2026-04-01"
today = datetime.strptime(today, "%Y-%m-%d").date()

# --- Validate 'today' ---
today = dv.validate_today(today)

DAYS_BACK = (today - start_date).days + 1

# Paths
BASE_DIR = Path("..") / "data"
removed_state = "Removed"

today_str = today.strftime("%Y-%m-%d")
output_file = BASE_DIR / f"{project}_Utilisation_{today_str}.xlsx"


# Load snapshots and sprints

In [ ]:
wi_raw_snapshots = cl.get_wi_snapshots(
    project=project,
    days_back=DAYS_BACK,
    removed_state=removed_state,
)

sprints, current_sprint, previous_sprint = cl.get_sprint_info(wi_raw_snapshots, today)


# Daily charge by task (input for utilisation)

In [ ]:
# Per-task daily charge is the bridge between work-item snapshots and the
# sprint each charge day actually belongs to.
daily_charge_by_task = dc.compute_daily_charge_by_task(wi_raw_snapshots)


## Utilisation calculation

### 2-1 Charged time mapped to sprints

In [ ]:
charge_by_sprint = ut.compute_charge_by_sprint(daily_charge_by_task, sprints)


### 2-2 Per-task utilisation detail (planned + unplanned)

In [ ]:
utilisation = ut.compute_utilisation_detail(
    wi_raw_snapshots=wi_raw_snapshots,
    sprints=sprints,
    charge_by_sprint=charge_by_sprint,
    start_date=start_date,
)


### 2-3 Per-(assignee, sprint) summary

In [ ]:
utilisation_summary = ut.compute_utilisation_summary(utilisation)


# Save Utilisation workbook

In [ ]:
xi.save_utilisation_report(
    output_file,
    utilisation_summary=utilisation_summary,
    utilisation=utilisation,
)

print(f"✅ Done! File saved: {output_file}")
